In [41]:
from typing import  Annotated,  List
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver
import operator as op
from dataclasses import dataclass, field
from typing import Annotated, List
import operator as op



In [42]:
@dataclass
class State:
    messages: Annotated[List, op.add] = field(default_factory=list)
    msg_count: int = 0


In [43]:
def chat_bot(state:State)-> dict:
    new_count = state.msg_count + 1
    return {
        "messages": [f"this is msg count {state.msg_count}"],
        "msg_count": new_count
    }

In [44]:
def build_graph(checkpointer=None):
    builder = StateGraph(State)
    
    builder.add_node("chat_bot", chat_bot)
    
    builder.add_edge(START, "chat_bot")
    builder.add_edge("chat_bot", END)
    
    graph = builder.compile(checkpointer = checkpointer)
    
    return graph

In [45]:
graph = build_graph(InMemorySaver())

config_a = {"configurable":{"thread_id":"a"}}
config_b = {"configurable":{"thread_id":"b"}}

msgs = [(config_a, {"messages":["msg 1 for a"]}),(config_b, {"messages":["msg 1 for b"]}),(config_a,{"messages":["msg 2 for a"]})]

for msg in msgs:
    res = graph.invoke(msg[1],msg[0])
    print(res)



{'messages': ['msg 1 for a', 'this is msg count 0'], 'msg_count': 1}
{'messages': ['msg 1 for b', 'this is msg count 0'], 'msg_count': 1}
{'messages': ['msg 1 for a', 'this is msg count 0', 'msg 2 for a', 'this is msg count 1'], 'msg_count': 2}


In [46]:
# Get state history
history_a = list(graph.get_state_history(config_a))
history_b = list(graph.get_state_history(config_b))
for snapshot in history_a:
  print(snapshot.config, snapshot.values)
  
for snapshot in history_b:
  print(snapshot.config, snapshot.values)

{'configurable': {'thread_id': 'a', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce8e2-fa81-6c7a-8004-a543f15f2936'}} {'messages': ['msg 1 for a', 'this is msg count 0', 'msg 2 for a', 'this is msg count 1'], 'msg_count': 2}
{'configurable': {'thread_id': 'a', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce8e2-fa7d-657f-8003-ec02920c25ae'}} {'messages': ['msg 1 for a', 'this is msg count 0', 'msg 2 for a'], 'msg_count': 1}
{'configurable': {'thread_id': 'a', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce8e2-fa7a-6e6a-8002-11fdd0013c9a'}} {'messages': ['msg 1 for a', 'this is msg count 0'], 'msg_count': 1}
{'configurable': {'thread_id': 'a', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce8e2-fa63-6cca-8001-6658dd5be565'}} {'messages': ['msg 1 for a', 'this is msg count 0'], 'msg_count': 1}
{'configurable': {'thread_id': 'a', 'checkpoint_ns': '', 'checkpoint_id': '1f1ce8e2-fa5d-67c7-8000-7a09a88afe8a'}} {'messages': ['msg 1 for a']}
{'configurable': {'thread_id': 'a', 'checkpoint_ns': '', 'checkpoi

In [49]:
# sqlite 

import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver 

# Setup connection to the same file
conn = sqlite3.connect("state_database.db", check_same_thread=False)
memory = SqliteSaver(conn)


In [50]:
graph = build_graph(memory)

config_a = {"configurable":{"thread_id":"a2"}}
config_b = {"configurable":{"thread_id":"b2"}}

msgs = [(config_a, {"messages":["msg 1 for a"]}),(config_b, {"messages":["msg 1 for b"]}),(config_a,{"messages":["msg 2 for a"]})]

for msg in msgs:
    res = graph.invoke(msg[1],msg[0])
    print(res)



{'messages': ['msg 1 for a', 'this is msg count 0'], 'msg_count': 1}
{'messages': ['msg 1 for b', 'this is msg count 0'], 'msg_count': 1}
{'messages': ['msg 1 for a', 'this is msg count 0', 'msg 2 for a', 'this is msg count 1'], 'msg_count': 2}
